# Lesson 10 Lab — Model Loading, Formats, and Provenance

**Puzzle:** Can a model name reproduce a deployment after its remote repository changes?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

A serving manifest must identify weight files, configuration, tokenizer, code trust, and revisions. A convenient repository name is mutable unless resolved to immutable content.


## 0. Predict before running

1. List the files required by this local checkpoint.
2. Predict whether one or multiple safetensors shards exist.
3. Choose the immutable identifiers for a release manifest.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

The lab audits the local checkpoint without network access: required files, safetensors header, configuration fields, tokenizer metadata, file sizes, and SHA-256 digests. It then confirms native loading in the pinned engine.

- Model weights and tokenizer are separate versioned artifacts.
- A format safety property is not a provenance record.
- Remote code changes the supply-chain boundary.


## 2. Derive the mechanism

vLLM combines a model configuration, tokenizer, weight loader, architecture implementation, and optional remote code. Safetensors avoids pickle execution but does not establish model license or semantic identity. Content hashes make local bytes immutable; upstream commit revisions make remote retrieval repeatable. `trust_remote_code` expands the executable trust boundary and must be an explicit decision.

### Mechanism at a glance

```mermaid
flowchart LR
  R["upstream revision"] --> M["local manifest"]
  W["safetensors bytes"] --> M
  C["config + tokenizer"] --> M
  T["remote-code trust decision"] --> M
  M --> L["vLLM loader"]
  L --> E["generation + signed evidence"]
```

### Walk it step by step

1. **Inventory artifacts.** Separate weights, config, tokenizer, and optional code.
2. **Resolve immutable identities.** Use commit revisions and content hashes.
3. **Declare trust.** Make remote-code and license decisions visible.
4. **Execute the manifest.** Prove the exact bytes load in the target engine.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 10
LESSON_TITLE = 'Model Loading, Formats, and Provenance'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260822
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | a mutable model name with implicit defaults |
| Candidate | a content-addressed local manifest plus native load |
| Held constant | checkpoint path, file bytes, offline mode, engine arguments, and prompt |
| Measurements | hashes, sizes, architecture, dtype, tokenizer class, trust setting, and load success |
| Evidence | `native-backend` |

**Experiment:** Hash the local model/config/tokenizer artifacts, inspect format metadata, and perform a native load/generation check.


## 5. Inspect the experiment code

The code reads JSON and safetensors metadata without deserializing arbitrary Python objects. Hashes are streamed so the 3 GB weight file does not enter host memory at once.

Do not execute until the code matches the frozen table.


In [2]:
cfg=model_config(); weights=sorted(MODEL.glob("*.safetensors"))
def stream_hash(paths):
    digest=hashlib.sha256()
    for path in paths:
        with path.open("rb") as handle:
            for chunk in iter(lambda:handle.read(8*2**20),b""): digest.update(chunk)
    return digest.hexdigest()
hashes={"config":hashlib.sha256((MODEL/"config.json").read_bytes()).hexdigest(),
        "tokenizer":hashlib.sha256((MODEL/"tokenizer.json").read_bytes()).hexdigest(),
        "weights":stream_hash(weights)}
llm=LLM(**base_engine_args(max_model_len=512)); output=llm.generate(["Say provenance."],
    SamplingParams(temperature=0.0,max_tokens=4),use_tqdm=False)[0]
metrics={"weight_files":len(weights),"weight_bytes":sum(p.stat().st_size for p in weights),"hashes":hashes,
         "architecture":str((cfg.get("architectures") or ["unknown"])[0]),
         "declared_dtype":str(cfg.get("torch_dtype")),
         "tokenizer_class":json.loads((MODEL/"tokenizer_config.json").read_text()).get("tokenizer_class"),
         "trust_remote_code":False,"native_load":bool(output.finished),"output":output_record(output)}
analysis=(f"The manifest covers {len(weights)} safetensors file(s), {metrics['weight_bytes']:,} bytes, "
          f"three hashes, and architecture {metrics['architecture']}. The exact bytes completed native generation.")


INFO 08-13 00:19:05 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260822, 'max_model_len': 512, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:19:05 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:19:05 [model.py:1883] Using max model len 512


INFO 08-13 00:19:05 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:19:05 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=650870) 2026-08-13 00:19:17,296 - INFO - autotuner.py:829 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=650870) 2026-08-13 00:19:17,355 - INFO - autotuner.py:852 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=650870) 2026-08-13 00:19:17,389 - INFO - autotuner.py:2269 - flashinfer.jit: [Autotuner]: Saved 0 configs to <remote-home>/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.16.post3/197e479b9af89d75faa86c9bc4f7a7286271d33da68ccb96d4465ab882f7faea/autotune_configs.json (0 new, 0 from previous config)


(EngineCore pid=650870) INFO 08-13 00:19:18 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=650870) INFO 08-13 00:19:18 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.70 s


(EngineCore pid=650870) WARNING 08-13 00:19:18 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=650870) WARNING 08-13 00:19:18 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=650870) INFO 08-13 00:19:18 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=650870) INFO 08-13 00:19:18 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=650870) INFO 08-13 00:19:18 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:19:19 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Weight files | 1 |
| Weight bytes | 3,087,467,144 bytes |
| Config hash | `98d2ff8cc474` |
| Tokenizer hash | `c0382117ea32` |
| Weight hash | `dd924a11b4c2` |
| Architecture | Qwen2ForCausalLM |
| Native load | yes |


## 7. Explain the result

The manifest covers 1 safetensors file(s), 3,087,467,144 bytes, three hashes, and architecture Qwen2ForCausalLM. The exact bytes completed native generation.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 10, "title": 'Model Loading, Formats, and Provenance', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Reproducible loading requires immutable bytes and explicit trust decisions; a model alias alone is insufficient.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 10,
  "title": "Model Loading, Formats, and Provenance",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260822
  },
  "evidence_label": "native-backend",
  "metrics": {
    "weight_files": 1,
    "weight_bytes": 3087467144,
    "hashes": {
      "config": "98d2ff8cc47488d08a2b0b3acf4eb99ef210779b42bd48605f6b8e36acdbf670",
      "tokenizer": "c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539",
      "weights": "dd924a11b4c220f385b51ffa522daea7c9f3d850e31b162bb5661df483c6d3ee"
    },
    "architecture": "Qwen2ForCausalLM",
    "declared_dtype": "bfloat16",
    "tokenizer_class": "Qwen2Tokenizer",
    "trust_remote_code": false,
    "native_load": true,
    "output": {
      "request_id": "0",
      "prompt_tokens": 4,
      "output_tokens": 4,
      "token_id

## 9. Make the bounded decision

> Reproducible loading requires immutable bytes and explicit trust decisions; a model alias alone is insufficient.

**Acceptance/rollback:** Release only when model, tokenizer, config, code trust, license review, and native load are bound to immutable identifiers.

**Failure analysis:** Local hashes cannot reveal the upstream commit if the directory lost repository metadata. A successful generation does not validate the license, training provenance, or every architecture feature.


## 10. Extend the evidence

Resolve the upstream commit, sign the manifest, verify it in image build and startup, and test a representative prompt suite after every loader change.

The full boundary and references are in [`README.md`](README.md).
